# 分词器

大模型不读英文，也不读中文。它读的是整数（词对应的ID等等）。分词器决定整数范围以及范围内整数携带的含义。

## 问题

每个词、空格、标点符号必须先被转换成一个整数，大模型才可以处理它。这种转换吧一些假设烙进大模型中，并且不可被撤销。

不合理的分词器直接导致相同的词需要更多的词元才能进行表示，进而导致上下文窗口严重缩水，且与大模型沟通的成本也会增加。

## 基本概念

目前为止有三种手段把词变为整数，其中两种不能够在大规模上运作。

### 词级分词 （Word-level Tokenization）

给每个词一个唯一的整数；这需要**十分庞大的词表**来装下所有你能想到的词，否则你会得到一个未知的（`[UNK]`）的词元，模型会说，我不知道这是什么。

### 字符级分词（Character-level Tokenization）

给每个字符一个唯一的整数；词表很小，但是表示一个词会消耗掉大量的词元，模型的注意力也会消耗在像“T-h-e” 这样三岁小孩都能理解的单词中。

### 子词分词（Subword Tokenization）

常见的词分配唯一的整数，没有见过/罕见的词则由常见的词组合而成。词表大小可控，分词后的序列长度也不会很长，没有见过的词也被自然的消化掉。是如今大模型的默认选择。

有以下几种形式：

#### 字节对编码（BPE：Byte Pair Encoding）

本质上是一种贪心压缩算法，从独立的字符集开始，将相邻两个字符出现频率高的一对进行合并，重复直到词表大小达到预期。

一开始的字符集加上按顺序合并的字节对表，就构成分词器。在编码一个新单词的时候，按字节对编码中的合并顺序进行合并。

合并结果与训练所使用的语料相关，所以说这些合并顺序的选择永久的改变了模型所能看到的内容～

#### WordPiece

与字节对编码类似，但是合并的时候不单单只看频率，而是最大化训练数据的似然（人话： 谁和谁合在一起，更说得通）。
```
BPE 的合并原则：         count(A, B)
WordPiece 的合并原则：   count(AB) / (count(A) * count(B))
```

所以BPE看的是哪个对出现的频率最高，而WordPiece看的是哪个对更倾向于一起出现。

同时WordPiece 还引入了前缀 `##` 作为连接符

```
unhappiness --> [un, ##happi, ##ness]
```

#### SentencePiece

直接在输入的Unicode字节流上切割，包括空格等，所以不会像BPE一样有将“hello world”拆分成“[hello, world]”的过程。所以不包含先验的语言类型及规则（是中文还是英文、是否采用空格分隔等等）。

两种算法：
1. BPE 模式。  与标准的BPE类似，只不过现在处理Unicode字符
2. Unigram 模型。  从一个巨大的词表出发，进行剪枝而非合并。

#### 词表大小的权衡

更大的词表意味着更多的参数；因为你需要为每个词训练、存储一个潜入向量。
同时更大的词表意味这一段文本，分词后词元数量更少（一些罕见词现在因为合并有了自己的ID），所以会更省钱，可能也更省算力。

## 坑

分词要考虑多语言。注意语料中不同语言的占比。

# 开始编码

In [5]:
from collections import Counter

class BPETokenizer:
    def __init__(self):
        self.merges = {}
        self.vocab = {}

    def _get_pairs(self, tokens):
        pairs = Counter()
        for pre, nxt in zip(tokens, tokens[1:]):
            pairs[(pre, nxt)] += 1
        return pairs

    def _merge_pair(self, tokens, best_pair, merge_token_id):
        merged = []
        i = 0
        while i < len(tokens) - 1:
            if tokens[i] == best_pair[0] and tokens[i + 1] == best_pair[1]:
                merged.append(merge_token_id)
                i += 2
            else:
                merged.append(tokens[i])
                i += 1
        return merged
            

    def train(self, text, num_merges):
        tokens = list(text.encode("utf-8"))
        self.vocab = {i: bytes([i]) for i in range(256)}
        merge_token_id = 256

        for _ in range(num_merges):
            pairs = self._get_pairs(tokens)
            if not pairs:
                break

            best_pair = max(pairs, key=pairs.get)

            tokens = self._merge_pair(tokens, best_pair, merge_token_id)
            self.merges[best_pair] = merge_token_id
            self.vocab[merge_token_id] = self.vocab[best_pair[0]] + self.vocab[best_pair[1]]

            merge_token_id += 1

        return self

    def encode(self, text):
        tokens = list(text.encode("utf-8"))
        for merge_pair, merge_token in self.merges.items():
            tokens = self._merge_pair(tokens, merge_pair, merge_token)
        
        return tokens

    def decode(self, tokens):
        byte_sequence = b"".join(self.vocab[t] for t in tokens)
        return byte_sequence.decode("utf-8", errors="replace")

    def vocab_size(self):
        return len(self.vocab)

In [6]:
import sys
from pathlib import Path

sys.path.append(str(Path("../../00_Common").resolve()))
from user_tools import SectionPrinter

with SectionPrinter("BPE Tokenizer"):
    text = "Hello world! This is a test."
    tokenizer = BPETokenizer()
    tokenizer.train(text, 10)
    encoded = tokenizer.encode(text)
    decoded = tokenizer.decode(encoded)
    print(f"Original: {text}")
    print(f"Encoded: {encoded}")
    print(f"Decoded: {decoded}")
    print(f"Vocab size: {tokenizer.vocab_size()}")

    print(f"Vocab: {tokenizer.vocab}")
    print(f"Merges: {tokenizer.merges}")


=======================BPE Tokenizer========================
Original: Hello world! This is a test.
Encoded: [265, 108, 100, 33, 32, 84]
Decoded: Hello world! T
Vocab size: 266
Vocab: {0: b'\x00', 1: b'\x01', 2: b'\x02', 3: b'\x03', 4: b'\x04', 5: b'\x05', 6: b'\x06', 7: b'\x07', 8: b'\x08', 9: b'\t', 10: b'\n', 11: b'\x0b', 12: b'\x0c', 13: b'\r', 14: b'\x0e', 15: b'\x0f', 16: b'\x10', 17: b'\x11', 18: b'\x12', 19: b'\x13', 20: b'\x14', 21: b'\x15', 22: b'\x16', 23: b'\x17', 24: b'\x18', 25: b'\x19', 26: b'\x1a', 27: b'\x1b', 28: b'\x1c', 29: b'\x1d', 30: b'\x1e', 31: b'\x1f', 32: b' ', 33: b'!', 34: b'"', 35: b'#', 36: b'$', 37: b'%', 38: b'&', 39: b"'", 40: b'(', 41: b')', 42: b'*', 43: b'+', 44: b',', 45: b'-', 46: b'.', 47: b'/', 48: b'0', 49: b'1', 50: b'2', 51: b'3', 52: b'4', 53: b'5', 54: b'6', 55: b'7', 56: b'8', 57: b'9', 58: b':', 59: b';', 60: b'<', 61: b'=', 62: b'>', 63: b'?', 64: b'@', 65: b'A', 66: b'B', 67: b'C', 68: b'D', 69: b'E', 70: b'F', 71: b'G', 72: b'H', 73: b

# 现成库

In [7]:
CORPUS = """
Natural language processing begins with tokenization, the process that converts raw text into discrete units a model can consume. Different designs trade vocabulary size against sequence length, and those choices permanently shape what the network sees during training and inference. Byte pair encoding starts from individual bytes or characters and repeatedly merges the most frequent adjacent pairs until a target vocabulary size is reached. The merge table becomes part of the tokenizer: encoding a new word replays those merges in

the same order that training discovered them. WordPiece also builds subword units, but it chooses merges by a likelihood-inspired score rather than raw co-occurrence counts alone. It marks continuation pieces with a special prefix so the model can distinguish word starts from mid-word fragments such as unhappiness becoming un and happi and ness. SentencePiece treats whitespace as an ordinary symbol and operates directly on Unicode, which makes the same pipeline usable for English, Chinese, and many other scripts without language-specific

pretokenization rules. Unigram language models reverse the BPE story by beginning with a large candidate vocabulary and pruning pieces that contribute least to the data likelihood. Large language models do not read letters or words in the human sense; they read integer token identifiers. A poorly designed tokenizer can explode the number of tokens needed for common phrases, shrink the effective context window, and raise serving cost for every prompt and completion. In practice engineers evaluate tokenizers on fertility, which

measures average tokens per word, on coverage of rare morphology, and on robustness to code, URLs, and multilingual mixtures. Balancing a larger embedding table against shorter sequences is a systems problem as much as a linguistic one. Consider a short story about a researcher who trains a small language model on diaries, manuals, and news articles. She watches the tokenizer invent compact pieces for frequent words like the and and, while rarer technical terms break into reusable stems and suffixes

that still preserve meaning across domains. Machine learning systems rely on careful data preparation, reproducible experiments, and clear evaluation metrics. Gradient descent adjusts parameters to reduce loss, while regularization methods such as dropout and weight decay help models generalize beyond the training set. Transformers attend over sequences with query, key, and value projections, enabling long-range dependencies without recurrence. Positional information can be added with sinusoidal encodings, learned embeddings, or relative schemes that better handle variable lengths. Software engineering for model

training involves datasets, dataloaders, checkpointing, mixed precision, and distributed strategies across multiple accelerators. Logging learning curves and inspecting failure cases often reveals tokenizer bugs long before architectural changes matter. Education materials explain that probability, linear algebra, and calculus form the mathematical backbone of modern deep learning. Students practice by implementing attention, residual connections, layer normalization, and feed-forward blocks from first principles. Open source communities share tokenizers, pretrained weights, and evaluation harnesses so practitioners can reproduce baselines and compare methods fairly.

Documentation that includes encoding examples and edge cases saves countless hours of debugging mysterious unknown tokens. Creative writing still matters for tokenizer corpora because fiction introduces dialogue, punctuation patterns, and narrative connectors that technical manuals underrepresent. Mixing genres produces merge rules that behave more stably when users switch between chat, code, and formal essays in the same session. Imagine cities connected by railways of information where each station is a token and each journey is a sentence. Compression algorithms decide

which stations to merge into express stops, leaving local stations for rare destinations that appear only occasionally in travel logs. Numerical text such as dates, measurements, and identifiers can fragment unpredictably if the training corpus never showed similar patterns. Including tables, formulas, and structured records encourages the tokenizer to keep useful digit groupings intact whenever possible. Multilingual corpora must respect script diversity: characters, syllables, and whitespace conventions vary widely. Underrepresenting a language forces that language into longer token sequences, which

quietly taxes latency and context for those users. During inference the model samples or searches over next-token distributions conditioned on previous identifiers. If tokenization is inconsistent between training and serving, the distribution the model learned no longer matches the inputs it receives, and quality collapses. Byte-level fallbacks guarantee that every Unicode string can be encoded without unknown symbols, at the cost of longer sequences for unfamiliar scripts. Hybrid designs combine a strong subword inventory with byte recovery for the long

tail of symbols and emojis. A practical exercise is to train a tiny BPE model on a few thousand words, inspect the earliest merges, and encode held-out sentences to see which fragments survive. Students usually discover that frequent function words become single tokens quickly, while long rare nouns remain compositional. Research papers discuss scaling laws, data quality, and alignment techniques that steer generative models toward helpful behavior. Yet even the strongest model remains constrained by the discrete vocabulary that sits

between human language and neural computation. The quick brown fox jumps over the lazy dog near the riverbank at dawn. Natural language processing begins with tokenization, the process that converts raw text into discrete units a model can consume. Scientists measure temperature, pressure, and humidity while calibrating sensitive laboratory instruments carefully. Programmers write tests, refactor modules, and review pull requests before merging changes into the main branch. Historians archive letters, maps, and photographs to reconstruct events that shaped communities over

centuries. SentencePiece treats whitespace as an ordinary symbol and operates directly on Unicode, which makes the same pipeline usable for English, Chinese, and many other scripts without language-specific pretokenization rules. Musicians rehearse melodies, harmonies, and rhythms until the ensemble performs with confidence and clarity. Farmers plant seeds, irrigate fields, and harvest crops according to seasonal weather patterns and soil conditions. Pilots check instruments, communicate with towers, and navigate routes across continents under changing skies. Consider a short story about a

researcher who trains a small language model on diaries, manuals, and news articles. Chefs prepare ingredients, balance flavors, and plate dishes that surprise guests with color and texture. Athletes train endurance, strength, and coordination through disciplined routines and recovery practices. Designers sketch interfaces, prototype interactions, and iterate layouts based on user feedback and analytics. Software engineering for model training involves datasets, dataloaders, checkpointing, mixed precision, and distributed strategies across multiple accelerators. The quick brown fox jumps over the lazy dog

near the riverbank at dawn. Scientists measure temperature, pressure, and humidity while calibrating sensitive laboratory instruments carefully. Programmers write tests, refactor modules, and review pull requests before merging changes into the main branch. Creative writing still matters for tokenizer corpora because fiction introduces dialogue, punctuation patterns, and narrative connectors that technical manuals underrepresent. Historians archive letters, maps, and photographs to reconstruct events that shaped communities over centuries. Musicians rehearse melodies, harmonies, and rhythms until the ensemble performs with confidence and

clarity. Farmers plant seeds, irrigate fields, and harvest crops according to seasonal weather patterns and soil conditions. Multilingual corpora must respect script diversity: characters, syllables, and whitespace conventions vary widely. Pilots check instruments, communicate with towers, and navigate routes across continents under changing skies. Chefs prepare ingredients, balance flavors, and plate dishes that surprise guests with color and texture. Athletes train endurance, strength, and coordination through disciplined routines and recovery practices. A practical exercise is to train a tiny BPE

model on a few thousand words, inspect the earliest merges, and encode held-out sentences to see which fragments survive. Designers sketch interfaces, prototype interactions, and iterate layouts based on user feedback and analytics. The quick brown fox jumps over the lazy dog near the riverbank at dawn. Scientists measure temperature, pressure, and humidity while calibrating sensitive laboratory instruments carefully. Byte pair encoding starts from individual bytes or characters and repeatedly merges the most frequent adjacent pairs until a target vocabulary

size is reached. Programmers write tests, refactor modules, and review pull requests before merging changes into the main branch. Historians archive letters, maps, and photographs to reconstruct events that shaped communities over centuries. Musicians rehearse melodies, harmonies, and rhythms until the ensemble performs with confidence and clarity. Large language models do not read letters or words in the human sense; they read integer token identifiers. Farmers plant seeds, irrigate fields, and harvest crops according to seasonal weather patterns and soil

conditions. Pilots check instruments, communicate with towers, and navigate routes across continents under changing skies. Chefs prepare ingredients, balance flavors, and plate dishes that surprise guests with color and texture. Machine learning systems rely on careful data preparation, reproducible experiments, and clear evaluation metrics. Athletes train endurance, strength, and coordination through disciplined routines and recovery practices. Designers sketch interfaces, prototype interactions, and iterate layouts based on user feedback and analytics. The quick brown fox jumps over the lazy dog near

the riverbank at dawn. Education materials explain that probability, linear algebra, and calculus form the mathematical backbone of modern deep learning. Scientists measure temperature, pressure, and humidity while calibrating sensitive laboratory instruments carefully. Programmers write tests, refactor modules, and review pull requests before merging changes into the main branch. Historians archive letters, maps, and photographs to reconstruct events that shaped communities over centuries. Imagine cities connected by railways of information where each station is a token and each journey is

a sentence. Musicians rehearse melodies, harmonies, and rhythms until the ensemble performs with confidence and clarity. Farmers plant seeds, irrigate fields, and harvest crops according to seasonal weather patterns and soil conditions. Pilots check instruments, communicate with towers, and navigate routes across continents under changing skies. During inference the model samples or searches over next-token distributions conditioned on previous identifiers. Chefs prepare ingredients, balance flavors, and plate dishes that surprise guests with color and texture. Athletes train endurance, strength, and

coordination through disciplined routines and recovery practices. Designers sketch interfaces, prototype interactions, and iterate layouts based on user feedback and analytics. Research papers discuss scaling laws, data quality, and alignment techniques that steer generative models toward helpful behavior. The quick brown fox jumps over the lazy dog near the riverbank at dawn. Scientists measure temperature, pressure, and humidity while calibrating sensitive laboratory instruments carefully. Programmers write tests, refactor modules, and review pull requests before merging changes into the main branch.

WordPiece also builds subword units, but it chooses merges by a likelihood-inspired score rather than raw co-occurrence counts alone. Historians archive letters, maps, and photographs to reconstruct events that shaped communities over centuries. Musicians rehearse melodies, harmonies, and rhythms until the ensemble performs with confidence and clarity. Farmers plant seeds, irrigate fields, and harvest crops according to seasonal weather patterns and soil conditions. In practice engineers evaluate tokenizers on fertility, which measures average tokens per word, on coverage of rare

morphology, and on robustness to code, URLs, and multilingual mixtures. Pilots check instruments, communicate with towers, and navigate routes across continents under changing skies. Chefs prepare ingredients, balance flavors, and plate dishes that surprise guests with color and texture. Athletes train endurance, strength, and coordination through disciplined routines and recovery practices. Transformers attend over sequences with query, key, and value projections, enabling long-range dependencies without recurrence. Designers sketch interfaces, prototype interactions, and iterate layouts based on user feedback and analytics.

The quick brown fox jumps over the lazy dog near the riverbank at dawn. Scientists measure temperature, pressure, and humidity while calibrating sensitive laboratory instruments carefully. Open source communities share tokenizers, pretrained weights, and evaluation harnesses so practitioners can reproduce baselines and compare methods fairly. Programmers write tests, refactor modules, and review pull requests before merging changes into the main branch. Historians archive letters, maps, and photographs to reconstruct events that shaped communities over centuries. Musicians rehearse melodies, harmonies, and

rhythms until the ensemble performs with confidence and clarity. Numerical text such as dates, measurements, and identifiers can fragment unpredictably if the training corpus never showed similar patterns. Farmers plant seeds, irrigate fields, and harvest crops according to seasonal weather patterns and soil conditions. Pilots check instruments, communicate with towers, and navigate routes across continents under changing skies. Chefs prepare ingredients, balance flavors, and plate dishes that surprise guests with color and texture. Byte-level fallbacks guarantee that every Unicode string

can be encoded without unknown symbols, at the cost of longer sequences for unfamiliar scripts. Athletes train endurance, strength, and coordination through disciplined routines and recovery practices. Designers sketch interfaces, prototype interactions, and iterate layouts based on user feedback and analytics. The quick brown fox jumps over the lazy dog near the riverbank at dawn. Natural language processing begins with tokenization, the process that converts raw text into discrete units a model can consume. Scientists measure temperature, pressure, and humidity

while calibrating sensitive laboratory instruments carefully. Programmers write tests, refactor modules, and review pull requests before merging changes into the main branch. Historians archive letters, maps, and photographs to reconstruct events that shaped communities over centuries. SentencePiece treats whitespace as an ordinary symbol and operates directly on Unicode, which makes the same pipeline usable for English, Chinese, and many other scripts without language-specific pretokenization rules. Musicians rehearse melodies, harmonies, and rhythms until the ensemble performs with confidence and clarity. Farmers

plant seeds, irrigate fields, and harvest crops according to seasonal weather patterns and soil conditions. Pilots check instruments, communicate with towers, and navigate routes across continents under changing skies. Consider a short story about a researcher who trains a small language model on diaries, manuals, and news articles. Chefs prepare ingredients, balance flavors, and plate dishes that surprise guests with color and texture. Athletes train endurance, strength, and coordination through disciplined routines and recovery practices. Designers sketch interfaces, prototype interactions,

and iterate layouts based on user feedback and analytics. Software engineering for model training involves datasets, dataloaders, checkpointing, mixed precision, and distributed strategies across multiple accelerators. The quick brown fox jumps over the lazy dog near the riverbank at dawn. Scientists measure temperature, pressure, and humidity while calibrating sensitive laboratory instruments carefully. Programmers write tests, refactor modules, and review pull requests before merging changes into the main branch. Creative writing still matters for tokenizer corpora because fiction introduces dialogue, punctuation

patterns, and narrative connectors that technical manuals underrepresent. Historians archive letters, maps, and photographs to reconstruct events that shaped communities over centuries. Musicians rehearse melodies, harmonies, and rhythms until the ensemble performs with confidence and clarity. Farmers plant seeds, irrigate fields, and harvest crops according to seasonal weather patterns and soil conditions. Multilingual corpora must respect script diversity: characters, syllables, and whitespace conventions vary widely. Pilots check instruments, communicate with towers, and navigate routes across continents under changing skies. Chefs

prepare ingredients, balance flavors, and plate dishes that surprise guests with color and texture. Athletes train endurance, strength, and coordination through disciplined routines and recovery practices. A practical exercise is to train a tiny BPE model on a few thousand words, inspect the earliest merges, and encode held-out sentences to see which fragments survive. Designers sketch interfaces, prototype interactions, and iterate layouts based on user feedback and analytics. The quick brown fox jumps over the lazy dog near the riverbank

at dawn. Scientists measure temperature, pressure, and humidity while calibrating sensitive laboratory instruments carefully. Byte pair encoding starts from individual bytes or characters and repeatedly merges the most frequent adjacent pairs until a target vocabulary size is reached. Programmers write tests, refactor modules, and review pull requests before merging changes into the main branch. Historians archive letters, maps, and photographs to reconstruct events that shaped communities over centuries. Musicians rehearse melodies, harmonies, and rhythms until the ensemble performs with confidence

and clarity. Large language models do not read letters or words in the human sense; they read integer token identifiers. Farmers plant seeds, irrigate fields, and harvest crops according to seasonal weather patterns and soil conditions. Pilots check instruments, communicate with towers, and navigate routes across continents under changing skies. Chefs prepare ingredients, balance flavors, and plate dishes that surprise guests with color and texture. Machine learning systems rely on careful data preparation, reproducible experiments, and clear evaluation metrics. Athletes

train endurance, strength, and coordination through disciplined routines and recovery practices. Designers sketch interfaces, prototype interactions, and iterate layouts based on user feedback and analytics. The quick brown fox jumps over the lazy dog near the riverbank at dawn. Education materials explain that probability, linear algebra, and calculus form the mathematical backbone of modern deep learning. Scientists measure temperature, pressure, and humidity while calibrating sensitive laboratory instruments carefully. Programmers write tests, refactor modules, and review pull requests before merging changes

into the main branch. Historians archive letters, maps, and photographs to reconstruct events that shaped communities over centuries. Imagine cities connected by railways of information where each station is a token and each journey is a sentence. Musicians rehearse melodies, harmonies, and rhythms until the ensemble performs with confidence and clarity. Farmers plant seeds, irrigate fields, and harvest crops according to seasonal weather patterns and soil conditions. Pilots check instruments, communicate with towers, and navigate routes across continents under changing

skies. During inference the model samples or searches over next-token distributions conditioned on previous identifiers. Chefs prepare ingredients, balance flavors, and plate dishes that surprise guests with color and texture. Athletes train endurance, strength, and coordination through disciplined routines and recovery practices. Designers sketch interfaces, prototype interactions, and iterate layouts based on user feedback and analytics. Research papers discuss scaling laws, data quality, and alignment techniques that steer generative models toward helpful behavior. The quick brown fox jumps over the

lazy dog near the riverbank at dawn. Scientists measure temperature, pressure, and humidity while calibrating sensitive laboratory instruments carefully. Programmers write tests, refactor modules, and review pull requests before merging changes into the main branch. WordPiece also builds subword units, but it chooses merges by a likelihood-inspired score rather than raw co-occurrence counts alone. Historians archive letters, maps, and photographs to reconstruct events that shaped communities over centuries. Musicians rehearse melodies, harmonies, and rhythms until the ensemble performs with confidence

and clarity. Farmers plant seeds, irrigate fields, and harvest crops according to seasonal weather patterns and soil conditions. In practice engineers evaluate tokenizers on fertility, which measures average tokens per word, on coverage of rare morphology, and on robustness to
"""
TEXT = "Hello world! This is a test."
print("CORPUS words:", len(CORPUS.split()))


CORPUS words: 3000


## Tiktoken -- GPT 预训练好的分词器

In [ ]:
import tiktoken

enc = tiktoken.get_encoding("gpt2")

tokens = enc.encode(TEXT)

print(tokens)

[198, 35364, 3303, 7587, 6140, 351, 11241, 1634, 11, 262, 1429, 326, 26161, 8246, 2420, 656, 28810, 4991, 257, 2746, 460, 15000, 13, 20615, 9824, 3292, 25818, 2546, 1028, 8379, 4129, 11, 290, 883, 7747, 15043, 5485, 644, 262, 3127, 7224, 1141, 3047, 290, 32278, 13, 30589, 5166, 21004, 4940, 422, 1981, 9881, 393, 3435, 290, 7830, 4017, 3212, 262, 749, 10792, 15909, 14729, 1566, 257, 2496, 25818, 2546, 318, 4251, 13, 383, 20121, 3084, 4329, 636, 286, 262, 11241, 7509, 25, 21004, 257, 649, 1573, 2186, 592, 883, 4017, 3212, 287, 198, 198, 1169, 976, 1502, 326, 3047, 5071, 606, 13, 9678, 47, 8535, 635, 12188, 850, 4775, 4991, 11, 475, 340, 19769, 4017, 3212, 416, 257, 14955, 12, 24194, 4776, 2138, 621, 8246, 763, 12, 13966, 33928, 9853, 3436, 13, 632, 8849, 24659, 5207, 351, 257, 2041, 21231, 523, 262, 2746, 460, 15714, 1573, 4940, 422, 3095, 12, 4775, 21441, 884, 355, 14274, 42661, 5033, 555, 290, 1147, 72, 290, 299, 408, 13, 11352, 594, 47, 8535, 18432, 13216, 10223, 355, 281, 8850, 6194,

## HUGGING FACE Tokenizers --- 训练自己的分词器时使用

In [10]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import ByteLevel

tokenizer = Tokenizer(BPE())
tokenizer.pre_tokenizer = ByteLevel()

trainer = BpeTrainer(vocab_size=500, special_tokens=["<pad>", "<eos>", "<unk>"])

# train() 要的是文件路径列表；内存字符串用 train_from_iterator
tokenizer.train_from_iterator([CORPUS], trainer=trainer)

output = tokenizer.encode(TEXT)

print(f"Output: {output.ids}")
print(f"Tokens: {output.tokens}")





Output: [327, 102, 39, 42, 77, 69, 39, 31, 305, 35, 90, 284, 108, 60, 58, 47, 5]
Tokens: ['ĠH', 'el', 'l', 'o', 'Ġw', 'or', 'l', 'd', 'ĠT', 'h', 'is', 'Ġis', 'Ġa', 'Ġt', 'es', 't', '.']


## HUGGING FACE Tokenizers --- 加载模型对应的分词器

In [11]:
from transformers import AutoTokenizer

MODEL_NAME = "gpt2"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print(tokenizer.encode(TEXT))



[15496, 995, 0, 770, 318, 257, 1332, 13]
